# Projet 2 — Student 1 : Market Analysis Agent (LSTM multi-actions)
Même préparation de données que le reste de l'équipe (`data_utils.py` : 20 actions, `SEQ_LEN = 20`, split chronologique 70/15/15 **par action**, cibles `Target_Up` et `Is_Safe`).
**Ajouts Student 1** : RSI(14) et MACD dans `TECH_COLS`. **Modèle** : un seul LSTM entraîné sur toutes les actions (pooled) qui prédit `Target_Up`.

> Fichiers requis dans le même dossier : `data_utils.py` et `data/stocks/<TICKER>.csv`

## 1. Imports et chargement des 20 actions

In [ ]:
import json, random, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from utils import *

random.seed(42); np.random.seed(42); tf.random.set_seed(42)

DATA_DIR = find_data_dir()
splits = {}
for t in TICKERS:
    try:
        splits[t] = load_and_split_stock(t, DATA_DIR)
    except FileNotFoundError:
        print("⚠️  fichier manquant, ignoré :", t)

print(f"{len(splits)} actions chargées :", list(splits))
pd.DataFrame({t: [len(a), len(b), len(c)] for t, (a, b, c) in splits.items()},
             index=["train", "val", "test"]).T

## 2. Aperçu des features (exemple : première action)

In [ ]:
first = list(splits)[0]
tr, va, te = splits[first]
print(first, "| train:", tr.Date.min().date(), "->", tr.Date.max().date(),
      "| test:", te.Date.min().date(), "->", te.Date.max().date())
tr[["Date", "Close"] + TECH_COLS + ["Target_Up", "Is_Safe"]].tail()

## 3. Normalisation et séquences
- `StandardScaler` ajusté **uniquement sur le train** (toutes actions réunies), puis appliqué à val/test
- séquences de 20 jours construites **à l'intérieur de chaque action**

In [ ]:
scaler = StandardScaler().fit(pd.concat([s[0][TECH_COLS] for s in splits.values()]))

def build(split_idx):
    Xs, ys, meta = [], [], []
    for t, parts in splits.items():
        d = parts[split_idx].copy()
        d[TECH_COLS] = np.clip(scaler.transform(d[TECH_COLS]), -5, 5)
        X, y = make_sequences(d, TECH_COLS, "Target_Up")
        Xs.append(X); ys.append(y); meta += [t] * len(y)
    return np.concatenate(Xs), np.concatenate(ys), np.array(meta)

X_train, y_train, _      = build(0)
X_val,   y_val,   _      = build(1)
X_test,  y_test,  m_test = build(2)
print("Train", X_train.shape, "| Val", X_val.shape, "| Test", X_test.shape)
print("Part de 'hausse' (train) : %.1f%%" % (100 * y_train.mean()))

## 4. Modèle LSTM

In [ ]:
model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, len(TECH_COLS))),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# Les séquences sont concaténées par action : on mélange les fenêtres du TRAIN (chaque fenêtre est déjà indépendante)
es = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=30, batch_size=128, callbacks=[es], verbose=2)

plt.figure(figsize=(9, 3.5))
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val")
plt.title("Loss"); plt.legend(); plt.show()

## 5. Évaluation (test)
Comparée à la baseline « toujours la classe majoritaire ». En bourse la direction journalière est très bruitée : ~50–55 % est un résultat réaliste.

In [ ]:
proba = model.predict(X_test, verbose=0).ravel()
pred = (proba > 0.5).astype(int)
base = max(y_test.mean(), 1 - y_test.mean())

print(f"Accuracy : {accuracy_score(y_test, pred):.3f}   (baseline {base:.3f})")
print(f"F1       : {f1_score(y_test, pred):.3f}")
print(f"AUC      : {roc_auc_score(y_test, proba):.3f}   (0.5 = hasard)")
print("Matrice de confusion :\n", confusion_matrix(y_test, pred))

rows = []
for t in splits:
    m = m_test == t
    if m.sum() > 0 and len(set(y_test[m])) > 1:
        rows.append((t, m.sum(), accuracy_score(y_test[m], pred[m]), roc_auc_score(y_test[m], proba[m])))
per_ticker = pd.DataFrame(rows, columns=["Ticker", "N_test", "Accuracy", "AUC"]).round(3)
per_ticker

## 6. Sortie de l'agent pour Student 3

In [ ]:
out = {}
for t, (tr, va, te) in splits.items():
    d = te.copy()
    d[TECH_COLS] = np.clip(scaler.transform(d[TECH_COLS]), -5, 5)
    window = d[TECH_COLS].values[-SEQ_LEN:].reshape(1, SEQ_LEN, len(TECH_COLS)).astype("float32")
    p_up = float(model.predict(window, verbose=0)[0, 0])
    last = te.iloc[-1]
    out[t] = {
        "agent": "market_analysis",
        "date": str(last["Date"].date()),
        "predicted_direction": "UP" if p_up > 0.5 else "DOWN",
        "probability_up": round(p_up, 4),
        "RSI_14": round(float(last["RSI_14"]) * 100, 2),
        "MACD_Norm": round(float(last["MACD_Norm"]), 5),
        "Volatility_7d": round(float(last["Volatility_7d"]), 5),
        "Volatility_21d": round(float(last["Volatility_21d"]), 5),
    }
print(json.dumps(out[first], indent=2))

json.dump(out, open("market_agent_output.json", "w"), indent=2)
model.save("models/agent1/lstm_market_agent.keras")
joblib.dump(scaler, "models/agent1/scaler_market.pkl")
print("Sauvegardé : market_agent_output.json, lstm_market_agent.keras, scaler_market.pkl")

## 7. Remarques pour le rapport
- Résultat proche du hasard = normal (marché quasi efficient) ; toujours comparer à la baseline.
- Pas de fuite de scaler (ajusté sur le train) ; fenêtres construites par action.
- Le split 70/15/15 est fait **par action** (convention d'équipe) : les périodes de test diffèrent d'une action à l'autre. Une alternative plus stricte serait des dates de coupure communes à toutes les actions.
- Le test inclut des périodes de forte volatilité (ex. mars 2020, COVID).